# 00 - Environment Setup & Sanity Checks
## XAI Skin Lesion Classification - Research Pipeline
**Purpose**: Verify your environment is ready before running any research experiments.

---
## Prerequisites: Python Environment

<details open>
<summary><strong>Step 1: Create Python 3.13 Virtual Environment</strong></summary>

Open a terminal in the `Skin_Lesion_GRADCAM_Classification` root directory:
```bash
py -3.13 -m venv .venv
```
</details>

<details>
<summary><strong>Step 2: Activate Virtual Environment</strong></summary>

```bash
# Git Bash / VS Code terminal
.venv/Scripts/activate

# PowerShell
.venv\Scripts\activate
```
</details>

<details>
<summary><strong>Step 3: Install PyTorch</strong></summary>

**For NVIDIA GPU (CUDA 12.4):**
```bash
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124
```

**For CPU only:**
```bash
pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
```
</details>

<details>
<summary><strong>Step 4: Register Jupyter Kernel</strong></summary>

```bash
pip install ipykernel
python -m ipykernel install --user --name=skin-lesion-env --display-name="Python 3.13"
```
</details>

<details>
<summary><strong>Step 5: Install Additional Dependencies</strong></summary>

```bash
pip install timm pandas matplotlib scikit-learn albumentations opencv-python
```
</details>

<details>
<summary><strong>Step 6: Verify Setup</strong></summary>

```python
import torch
print(f"PyTorch:      {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU")
```
</details>

<details>
<summary><strong>Selecting Kernel in VS Code</strong></summary>

1. Install the **Jupyter** extension by Microsoft in VS Code
2. Open this notebook file in VS Code
3. Click "Select Kernel" in the top-right corner
4. Choose "Python 3.13" from the list
5. Run cells with the play button or `Shift+Enter`
</details>

<details>
<summary><strong>Troubleshooting</strong></summary>

| Issue | Solution |
|-------|----------|
| `Access is denied` when creating venv | Delete `.venv` folder and retry |
| PyTorch install fails | Try `cu121` index instead: `--index-url https://download.pytorch.org/whl/cu121` |
| No GPU available | That's fine - PyTorch runs on CPU automatically |
| Module not found | Run `pip install` again in the activated venv |
</details>

---
## How to Run This Notebook
### Option 1: Run from VS Code (Recommended)
1. Install the **Jupyter** extension by Microsoft in VS Code
2. Open this notebook file in VS Code
3. Select your Python kernel:
   - Click "Select Kernel" in the top-right corner
   - Or press `Ctrl+Shift+P` - "Notebook: Notebook Operations" - "Change Cell Execution Kernel"
4. Run cells with the play button or `Shift+Enter`

### Option 2: Run from PyCharm
1. Install the **Pythonid** plugin (for .ipynb support)
2. Right-click this file - "Open As Notebook"
3. Configure the Python interpreter to point to your virtualenv

### Option 3: Standalone Jupyter Server
```bash
# Navigate to notebooks directory
cd Skin_Lesion_Classification_frontend/notebooks

# Start Jupyter
jupyter lab
# Or: jupyter notebook
```
---
## CELL 0: Python Environment Setup
**WHAT THIS DOES**: Detects your Python environment and ensures the backend code is on the path.
**WHY**: The notebooks import from `../Skin_Lesion_Classification_backend/ml/` so they need the correct working directory.

In [1]:
import os
import sys
from pathlib import Path

# Get the notebook's directory
NOTEBOOK_DIR = Path(os.getcwd())
BACKEND_DIR = NOTEBOOK_DIR.parent.parent / "Skin_Lesion_Classification_backend"
ML_DIR = BACKEND_DIR / "ml"

# Add backend ml/ to sys.path so imports like `from src.data.dataset` work
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Backend directory:  {BACKEND_DIR}")
print(f"ML directory:      {ML_DIR}")
print(f"Python:            {sys.executable}")
print(f"sys.path[0]:       {sys.path[0]}")

# Verify backend exists
if not BACKEND_DIR.exists():
    print(f"\u274c ERROR: Backend not found at {BACKEND_DIR}")
    print("Make sure the backend repo is cloned at the same level as the frontend repo.")
else:
    print("\u2705 Backend directory found.")

Notebook directory: c:\Users\saiyu\Desktop\projects\KI_projects\Skin_Lesion_GRADCAM_Classification\Skin_Lesion_Classification_frontend\notebooks
Backend directory:  c:\Users\saiyu\Desktop\projects\KI_projects\Skin_Lesion_GRADCAM_Classification\Skin_Lesion_Classification_backend
ML directory:      c:\Users\saiyu\Desktop\projects\KI_projects\Skin_Lesion_GRADCAM_Classification\Skin_Lesion_Classification_backend\ml
Python:            c:\Users\saiyu\AppData\Local\Programs\Python\Python313\python.exe
sys.path[0]:       c:\Users\saiyu\Desktop\projects\KI_projects\Skin_Lesion_GRADCAM_Classification\Skin_Lesion_Classification_backend\ml
✅ Backend directory found.


---

## CELL 1: Environment Sanity Check

**WHY**: Catch environment problems before wasting hours training.

Run this first on any new machine or Python environment.

In [ ]:
import torch
import torchvision
import timm
import numpy as np
import pandas as pd
import matplotlib
import sklearn
import cv2
import albumentations

print("=== Environment Check ===")
print(f"PyTorch:     {torch.__version__}")
print(f"Torchvision: {torchvision.__version__}")
print(f"timm:        {timm.__version__}")
print(f"NumPy:       {np.__version__}")
print(f"OpenCV:      {cv2.__version__}")
print(f"Scikit-learn:{sklearn.__version__}")

print(f"\n=== GPU ===")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
    print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    # Quick compute test
    a = torch.randn(1000, 1000).cuda()
    b = torch.randn(1000, 1000).cuda()
    c = a @ b
    print(f"Matrix multiply test: PASSED")
else:
    print("WARNING: No GPU - training will be VERY slow. Consider Google Colab.")

c:\Users\saiyu\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== Environment Check ===
PyTorch:     2.6.0+cu124
Torchvision: 0.21.0+cu124
timm:        1.0.26
NumPy:       2.4.3
OpenCV:      4.13.0
Scikit-learn:1.8.0

=== GPU ===
CUDA available: True
GPU:            NVIDIA GeForce RTX 4070 Laptop GPU
VRAM:           8.6 GB
Matrix multiply test: PASSED


---

## CELL 2: Data Sanity Check

**WHY**: Confirm data loaded correctly before training.

**Prerequisite**: You must have downloaded the HAM10000 dataset and run the data pipeline.
The backend repo should contain:
- `ml/data/processed/metadata_with_paths.csv` - generated metadata
- `ml/data/raw/` - original HAM10000 images

If `metadata_with_paths.csv` does not exist, you need to download HAM10000 and run the data pipeline:
```bash
cd Skin_Lesion_Classification_backend
python ml/scripts/process_ham10000.py  # or whatever the data pipeline script is
```

In [ ]:
import pandas as pd
from pathlib import Path

# Point to backend data directory
DATA_DIR = ML_DIR / "data" / "processed"
METADATA_PATH = DATA_DIR / "metadata_with_paths.csv"

if not METADATA_PATH.exists():
    print(f"\u274c ERROR: Metadata not found at {METADATA_PATH}")
    print("Download HAM10000 and run the data pipeline first.")
    print("See: https://github.com/ham1000/skin-cancer-mnist-ham10000")
    df = None
else:
    df = pd.read_csv(METADATA_PATH)
    print(f"\u2705 Metadata loaded: {METADATA_PATH}")

if df is not None:
    print(f"\n=== Dataset Overview ===")
    print(f"Total images:      {len(df)}")
    print(f"Unique patients:   {df['patient_id'].nunique()}")
    print(f"Unique lesion types: {df['dx'].nunique()}")
    print(f"\nClass distribution:")
    print(df['dx'].value_counts())
    print(f"\nBinary label distribution:")
    print(df['label_name'].value_counts())
    print(f"\nImbalance ratio: {df['label'].value_counts()[0] / df['label'].value_counts()[1]:.2f}:1 (benign:malignant)")
    
    # Check for missing files
    missing = df[df['filepath'].isna()]
    print(f"\nMissing image files: {len(missing)}")
    
    # Check image is actually readable
    import random
    from PIL import Image
    sample_paths = df['filepath'].dropna().sample(5, random_state=42).tolist()
    for p in sample_paths:
        try:
            img = Image.open(p)
            print(f"OK: {Path(p).name} - {img.size} {img.mode}")
        except Exception as e:
            print(f"BROKEN: {p} - {e}")

---

## CELL 3: Data Split Leakage Check

**WHY**: Patient leakage is the #1 mistake in medical ML papers.
If the same patient appears in train AND test, your results are artificially inflated.
This test proves you have no leakage.

**Expected result**: All overlap counts must be 0.

In [ ]:
if df is None:
    print("Skipping - no data loaded in Cell 2.")
else:
    from src.data.dataset import create_splits
    
    train_df, val_df, test_df = create_splits(df)
    
    print("=== Split Sizes ===")
    print(f"Train: {len(train_df)} images, {train_df['patient_id'].nunique()} patients")
    print(f"Val:   {len(val_df)} images,   {val_df['patient_id'].nunique()} patients")
    print(f"Test:  {len(test_df)} images,  {test_df['patient_id'].nunique()} patients")
    
    # THE CRITICAL TEST - must all be 0
    train_patients = set(train_df['patient_id'])
    val_patients   = set(val_df['patient_id'])
    test_patients  = set(test_df['patient_id'])
    
    overlap_tv = train_patients & val_patients
    overlap_tt = train_patients & test_patients
    overlap_vt = val_patients & test_patients
    
    print(f"\n=== Leakage Check ===")
    print(f"Train/Val overlap:  {len(overlap_tv)} patients  \u2190 must be 0")
    print(f"Train/Test overlap: {len(overlap_tt)} patients  \u2190 must be 0")
    print(f"Val/Test overlap:   {len(overlap_vt)} patients  \u2190 must be 0")
    
    if len(overlap_tv) == 0 and len(overlap_tt) == 0 and len(overlap_vt) == 0:
        print("\n\u2705 No data leakage. Split is valid.")
    else:
        print("\n\u274c LEAKAGE DETECTED. Fix create_splits() before training!")
    
    # Check class balance in each split
    for name, split_df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
        ratio = split_df['label'].mean()
        print(f"{name} malignant ratio: {ratio:.3f} ({ratio*100:.1f}%)")

---

## CELL 4: DataLoader Sanity Check

**WHY**: Confirm transforms work, tensor shapes are correct,
and WeightedRandomSampler is actually balancing the classes.

In [ ]:
if df is None:
    print("Skipping - no data loaded in Cell 2.")
else:
    from src.data.dataset import get_dataloaders
    import matplotlib.pyplot as plt
    import numpy as np
    
    train_loader, val_loader, test_loader, splits = get_dataloaders(
        df, batch_size=8, img_size=224, num_workers=0
    )
    
    # Get one batch
    images, labels, ids = next(iter(train_loader))
    print(f"Image tensor shape: {images.shape}")   # (8, 3, 224, 224)
    print(f"Label tensor shape: {labels.shape}")   # (8,)
    print(f"Label values:       {labels.tolist()}")
    print(f"Value range:        [{images.min():.3f}, {images.max():.3f}]")  # Should be approx [-2, 2] after normalize
    
    # Visualize - ALWAYS look at your data!
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    # De-normalize for display (ImageNet mean/std)
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    
    for i in range(8):
        ax = axes[i // 4][i % 4]
        img = images[i].permute(1, 2, 0).numpy()
        img = img * std + mean  # de-normalize
        img = np.clip(img, 0, 1)
        ax.imshow(img)
        ax.set_title(f"{'Malignant' if labels[i] == 1 else 'Benign'}", 
                     color='red' if labels[i] == 1 else 'green', fontweight='bold')
        ax.axis('off')
    
    plt.suptitle('Training Batch (verify augmentations look reasonable)', fontsize=14)
    plt.tight_layout()
    
    # Save figure relative to notebook directory
    OUTPUTS_DIR = NOTEBOOK_DIR / "outputs" / "figures"
    OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
    plt.savefig(OUTPUTS_DIR / "sanity_training_batch.png", dpi=150)
    print(f"\nFigure saved to: {OUTPUTS_DIR / 'sanity_training_batch.png'}")
    plt.show()
    
    # Check WeightedSampler is working by counting labels over many batches
    print("\n=== Sampler Balance Check (should be ~50/50) ===")
    label_counts = {0: 0, 1: 0}
    for _, labels_batch, _ in train_loader:
        for l in labels_batch.tolist():
            label_counts[l] += 1
    total = sum(label_counts.values())
    print(f"Benign:    {label_counts[0]} ({label_counts[0]/total*100:.1f}%)")
    print(f"Malignant: {label_counts[1]} ({label_counts[1]/total*100:.1f}%)")
    print("If benign >> malignant, your sampler is broken!")

---

## CELL 5: Verify Model Checkpoints Exist

**WHY**: Most research notebooks require trained model weights.
Check that they exist before running experiments.

In [ ]:
from pathlib import Path

MODEL_DIR = ML_DIR / "outputs" / "models"
CKPT_DIR  = ML_DIR / "outputs" / "models" / "checkpoints"

print("=== Model Checkpoints ===")
for p in sorted(MODEL_DIR.glob("*.pth")):
    size_mb = p.stat().st_size / 1e6
    print(f"  {p.name} ({size_mb:.1f} MB)")

print("\n=== Training Checkpoints (for RQ5) ===")
for p in sorted(CKPT_DIR.glob("*.pth")):
    print(f"  {p.name}")

if not list(MODEL_DIR.glob("*.pth")):
    print("\n\u26a0\ufe0f No model checkpoints found. Train models before running RQ notebooks.")
    print("    See the training guide in the backend repo.")
else:
    print("\n\u2705 Model checkpoints found - you can run RQ notebooks.")